# Brainstorm CTF Phantom: `invert` Solver Benchmark

Evaluates solvers on the [Brainstorm CTF phantom dataset](https://mne.tools/stable/auto_tutorials/inverse/85_brainstorm_phantom_ctf.html).

- Single dipole at **[18, 0, 49] mm**, stimulated at 23 Hz
- Two recordings: **20 µA** (low SNR) and **200 µA** (high SNR)
- Empty room recording for noise covariance

In [1]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mne.datasets.brainstorm import bst_phantom_ctf

from invert import Solver

## Config

In [2]:
SOLVER_NAME = "LCMV"
SOLVER_KWARGS = {}

ACTUAL_POS = np.array([18.0, 0.0, 49.0]) / 1000.0  # meters
DIP_FREQ = 23.0  # Hz

HEAD_RADIUS = 0.080
SOURCE_RADIUS = 0.060  # tighter than head to avoid shallow source bias
SOURCE_GRID_MM = 7.0
MINDIST_MM = 5.0

## Load data

In [3]:
data_path = bst_phantom_ctf.data_path(verbose=False)

def load_recording(name):
    raw = mne.io.read_raw_ctf(str(data_path / name), preload=True, verbose=False)
    sinusoid, _ = raw[raw.ch_names.index('HDAC006-4408')]
    crossings = np.where(np.diff(sinusoid > 0.5) > 0)[1] + raw.first_samp
    crossings = crossings[::2]  # 46 Hz crossings -> 23 Hz events
    events = np.zeros((len(crossings), 3), dtype=int)
    events[:, 0] = crossings
    events[:, 2] = 1
    epochs = mne.Epochs(
        raw, events, tmin=-0.5/DIP_FREQ, tmax=0.5/DIP_FREQ,
        baseline=(None, None), preload=True, verbose=False,
    )
    return epochs, epochs.average()

epochs_20, evoked_20 = load_recording('phantom_20uA_20150603_03.ds')
epochs_200, evoked_200 = load_recording('phantom_200uA_20150709_01.ds')

# Noise covariance from empty room
raw_er = mne.io.read_raw_ctf(
    str(data_path / 'emptyroom_20150709_01.ds'), preload=True, verbose=False,
)
noise_cov = mne.compute_raw_covariance(raw_er, verbose=False)

print(f'20 µA: {len(epochs_20)} epochs')
print(f'200 µA: {len(epochs_200)} epochs')

/Users/lukas/projects/invert/invert-package/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


20 µA: 229 epochs
200 µA: 700 epochs


## Forward model

In [ ]:
sphere

<ConductorModel | Sphere (3 layers): r0=[0.0, 0.0, 0.0] R=80 mm>

In [4]:
def make_forward(info):
    sphere = mne.make_sphere_model(r0=(0., 0., 0.), head_radius=HEAD_RADIUS, verbose=False)
    src_grid = mne.setup_volume_source_space(
        subject=None, pos=SOURCE_GRID_MM,
        sphere=(0., 0., 0., SOURCE_RADIUS),
        mindist=MINDIST_MM, exclude=0.0, verbose=False,
    )
    rr_used = src_grid[0]['rr'][src_grid[0]['vertno']]
    rr = np.vstack([rr_used, ACTUAL_POS])
    rr = np.unique(np.round(rr, 6), axis=0)
    norms = np.linalg.norm(rr, axis=1, keepdims=True)
    nn = np.zeros_like(rr)
    mask = norms.squeeze() > 0
    nn[mask] = rr[mask] / norms[mask]
    nn[~mask] = [1., 0., 0.]
    src = mne.setup_volume_source_space(
        subject=None, pos={'rr': rr, 'nn': nn},
        sphere=(0., 0., 0., SOURCE_RADIUS),
        mindist=0.0, exclude=0.0, verbose=False,
    )
    fwd = mne.make_forward_solution(
        info, trans=None, src=src, bem=sphere,
        meg=True, eeg=False, verbose=False,
    )
    return fwd, sphere

fwd_20, sphere = make_forward(evoked_20.info)
fwd_200, _ = make_forward(evoked_200.info)

pos = np.concatenate([s['rr'][s['vertno']] for s in fwd_20['src']], axis=0)
print(f'Source space: {pos.shape[0]} points')

Source space: 2008 points


## MNE dipole fit baseline

In [5]:
for label, evoked in [('20 µA', evoked_20), ('200 µA', evoked_200)]:
    dip = mne.fit_dipole(evoked.copy().crop(0, 0), noise_cov, sphere, verbose=False)[0]
    err = 1e3 * np.linalg.norm(dip.pos[0] - ACTUAL_POS)
    print(f'{label}: {err:.1f} mm (GOF {dip.gof[0]:.1f}%)')

20 µA: 7.7 mm (GOF 58.9%)
200 µA: 4.7 mm (GOF 97.7%)


## Evaluate solver

In [6]:
def evaluate_solver(solver_name, evoked, epochs, fwd, noise_cov, **kwargs):
    solver = Solver(solver_name, **kwargs)
    is_bf = getattr(solver, '_is_beamformer', False)
    mne_obj = epochs if is_bf else evoked
    solver.make_inverse_operator(fwd, mne_obj, noise_cov=noise_cov)
    stc = solver.apply_inverse_operator(evoked.copy().crop(0, 0))

    power = np.max(np.abs(stc.data), axis=1)
    n_src = pos.shape[0]
    if power.shape[0] != n_src and power.shape[0] % n_src == 0:
        power = np.max(power.reshape(n_src, power.shape[0] // n_src), axis=1)
    peak_idx = int(np.argmax(power))
    err_mm = 1e3 * np.linalg.norm(pos[peak_idx] - ACTUAL_POS)
    return err_mm, stc, solver

for label, evoked, epochs, fwd in [
    ('20 µA', evoked_20, epochs_20, fwd_20),
    ('200 µA', evoked_200, epochs_200, fwd_200),
]:
    err, stc, solver = evaluate_solver(
        SOLVER_NAME, evoked, epochs, fwd, noise_cov, **SOLVER_KWARGS,
    )
    print(f'{label} — {SOLVER_NAME}: {err:.1f} mm')

Removing 5 compensators from info because not all compensation channels were picked.
Removing 5 compensators from info because not all compensation channels were picked.
20 µA — LCMV: 51.0 mm
Removing 5 compensators from info because not all compensation channels were picked.
Removing 5 compensators from info because not all compensation channels were picked.
200 µA — LCMV: 23.7 mm


## Compare multiple solvers

In [7]:
SOLVERS = [
    'eLORETA', 'sLORETA', 'SR-eLORETA',
    'ESMV', 'ESMV-MVPURE', 'Flex-ESMV', 'LCMV',
    'MNE', 'wMNE', 'dSPM',
    'Champagne', 'Subspace-SBL',
    'RAP-MUSIC', 'FLEX-MUSIC', 'OMP',
    'ECD',
]

rows = []
for sname in SOLVERS:
    for label, evoked, epochs, fwd in [
        ('20uA', evoked_20, epochs_20, fwd_20),
        ('200uA', evoked_200, epochs_200, fwd_200),
    ]:
        try:
            err, _, _ = evaluate_solver(sname, evoked, epochs, fwd, noise_cov)
            rows.append({'solver': sname, 'recording': label, 'error_mm': err})
        except Exception as e:
            rows.append({'solver': sname, 'recording': label, 'error_mm': float('nan')})
            print(f'{sname} / {label}: FAILED — {e}')

df = pd.DataFrame(rows)
pivot = df.pivot(index='solver', columns='recording', values='error_mm')
pivot['mean'] = pivot.mean(axis=1)
pivot = pivot.sort_values('mean')
print(pivot.to_string(float_format='%.1f'))

Removing 5 compensators from info because not all compensation channels were picked.
Removing 5 compensators from info because not all compensation channels were picked.


GCV selected the lowest regularization parameter for Exact Low Resolution Tomography in the search range (idx=0 of 9, alpha=6.842212e-09). Consider widening or shifting `r_values`.


Removing 5 compensators from info because not all compensation channels were picked.
Removing 5 compensators from info because not all compensation channels were picked.
Removing 5 compensators from info because not all compensation channels were picked.
Removing 5 compensators from info because not all compensation channels were picked.


GCV selected the lowest regularization parameter for Standardized Low Resolution Tomography in the search range (idx=0 of 9, alpha=1.014225e-08). Consider widening or shifting `r_values`.


Removing 5 compensators from info because not all compensation channels were picked.
Removing 5 compensators from info because not all compensation channels were picked.
Removing 5 compensators from info because not all compensation channels were picked.
Removing 5 compensators from info because not all compensation channels were picked.


...

Removing 5 compensators from info because not all compensation channels were picked.
Removing 5 compensators from info because not all compensation channels were picked.


GCV selected the lowest regularization parameter for Weighted Minimum Norm Estimate in the search range (idx=0 of 9, alpha=5.890750e+01). Consider widening or shifting `r_values`.


Removing 5 compensators from info because not all compensation channels were picked.
Removing 5 compensators from info because not all compensation channels were picked.
Removing 5 compensators from info because not all compensation channels were picked.
Removing 5 compensators from info because not all compensation channels were picked.


GCV selected the lowest regularization parameter for Dynamic Statistical Parametric Mapping in the search range (idx=0 of 9, alpha=2.101686e+10). Consider widening or shifting `r_values`.


Removing 5 compensators from info because not all compensation channels were picked.
Removing 5 compensators from info because not all compensation channels were picked.
Removing 5 compensators from info because not all compensation channels were picked.
Removing 5 compensators from info because not all compensation channels were picked.
Removing 5 compensators from info because not all compensation channels were picked.
Removing 5 compensators from info because not all compensation channels were picked.
Removing 5 compensators from info because not all compensation channels were picked.
Removing 5 compensators from info because not all compensation channels were picked.
Removing 5 compensators from info because not all compensation channels were picked.
Removing 5 compensators from info because not all compensation channels were picked.
Removing 5 compensators from info because not all compensation channels were picked.
Removing 5 compensators from info because not all compensation ch